# Data preparation

Assembles the six-asset OHLCV panel (five index/commodity series from Yahoo Finance plus JP Morgan from CRSP), saves the close-to-close price panel `indices.csv`, and computes several realized volatility (RV) estimators. All data paths are managed through `file_manager`.

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from utils.file_manager import quantDiffusionPath

file_manager = quantDiffusionPath()   # all data paths are managed through file_manager

## Helper functions

In [ ]:
def fill_zero_rv_with_sampling(df):
    df_filled = df.copy()

    for col in df.columns:
        zero_mask = (df[col] == 0)
        num_zeros = zero_mask.sum()

        if num_zeros > 0:
            # keep only the non-zero values
            non_zero_values = df.loc[~zero_mask, col].values
            # resample with replacement from the non-zero values
            random_samples = np.random.choice(non_zero_values, size=num_zeros)
            df_filled.loc[zero_mask, col] = random_samples

    print("Filled zero values using each asset's existing distribution.")
    return df_filled


# Compute several realized volatility (RV) estimators
def calculate_rvs(O, H, L, C):
    rv_dict = {}

    # (1) Parkinson (1980) - high-low based
    # Formula: 1 / (4 * ln2) * (ln(H/L))^2
    rv_dict['Parkinson'] = (1 / (4 * np.log(2))) * (np.log(H / L)**2)

    # (2) Garman-Klass (1980) - OHLC based
    # Formula: 0.5 * (ln(H/L))^2 - (2*ln2 - 1) * (ln(C/O))^2
    rv_dict['Garman_Klass'] = 0.5 * (np.log(H / L)**2) - (2 * np.log(2) - 1) * (np.log(C / O)**2)

    # (3) Rogers-Satchell (1991) - drift independent
    # Formula: ln(H/C)*ln(H/O) + ln(L/C)*ln(L/O)
    rv_dict['Rogers_Satchell'] = np.log(H / C) * np.log(H / O) + np.log(L / C) * np.log(L / O)

    # (4) Simple close-to-close volatility (for comparison)
    # squared daily log returns
    returns = np.log(C / C.shift(1))
    rv_dict['Close_to_Close'] = returns**2

    return rv_dict

## 1. Assemble the six-asset panel and save `indices.csv`

In [ ]:
# JPM (CRSP) OHLCV
df_JPM = pd.read_csv(file_manager.get_jpm_file()).set_index('DlyCalDt').sort_index()
df_JPM = df_JPM[['DlyOpen', 'DlyHigh', 'DlyLow', 'DlyClose', 'DlyVol']]
df_JPM.columns = ['O', 'H', 'L', 'C', 'V']

# Combine the raw index OHLCV (data/indices/) with JPM into a six-asset panel
data_dict = {}
for field in ["O", "H", "L", "C", "V"]:
    df = pd.read_csv(file_manager.get_indices_ohlcv_file(field), index_col=0)
    df = df[["^DJI", "^IXIC", "^HSI", "GC=F", "CL=F"]]
    df = pd.concat([df, df_JPM[field]], axis=1)
    df.columns = ["DJI", "NASDAQ", "HSI", "GOLD", "WTI_OIL", "JPM"]
    df = df[["DJI", "NASDAQ", "JPM", "HSI", "GOLD", "WTI_OIL"]]
    df = df.dropna()
    df = df.loc["2001-01-02":"2025-12-31", :]
    df.index = pd.to_datetime(df.index)
    data_dict[field] = df

# Assembled close-price panel = new indices.csv (index closes + JPM)
indices_df = data_dict["C"].copy()
indices_df.index.name = "Date"
indices_df.to_csv(file_manager.get_indices_file())
print(f"saved {file_manager.get_indices_file()}  shape={indices_df.shape}  "
      f"range={indices_df.index.min().date()} ~ {indices_df.index.max().date()}")

## 2. Realized volatility estimators

In [ ]:
# Base OHLC data (from the assembled data_dict)
O = data_dict['O']
H = data_dict['H']
L = data_dict['L']
C = data_dict['C']

# Compute the RV estimators
rv_results = calculate_rvs(O, H, L, C)

In [ ]:
os.makedirs(file_manager.get_data_dir("RV"), exist_ok=True)
for rv_name in rv_results.keys():
    print(rv_name)
    rv_results[rv_name] = fill_zero_rv_with_sampling(rv_results[rv_name])
    rv_results[rv_name].to_csv(file_manager.get_rv_file(rv_name))

## 6. `our_basket`: five evaluation baskets from REAL log returns

Same static-portfolio construction as TailGAN's `Gen_StaticPort`, with **one change**: the
transition-matrix entries are sampled from the **real log returns** (TRAIN period for the IS
matrix, TEST period for the OOS matrix) rather than a fake standard normal. Density, column
scaling, the IsStock filter and `price_start` are unchanged. Notebooks 1 and 2 evaluate
portfolio VaR/ES against these baskets' OOS matrices.

In [ ]:
# ===== our_basket: 5 baskets, transition-matrix entries drawn from REAL log returns =====
# TailGAN Gen_StaticPort uses data_rvs = standard normal (fake). Here the only change is
# data_rvs = the real log returns: TRAIN -> TransMat_IS, TEST -> TransMat_OOS. Same logic.
import shutil
from scipy import sparse

_N_STOCKS, _N_PORTS = 6, 50
_ind = pd.read_csv("./data/indices.csv")
_ind["Date"] = pd.to_datetime(_ind["Date"], format="%Y-%m-%d")
_ind = _ind.set_index("Date").apply(pd.to_numeric).astype(float)
_lr = np.log1p(_ind.pct_change()).dropna().values          # (T, 6) real log returns
_cut = int(0.7 * len(_lr))                                  # chronological 70/30 split
_train_flat = _lr[:_cut].reshape(-1)                       # real TRAIN returns (pooled)
_test_flat = _lr[_cut:].reshape(-1)                        # real TEST  returns (pooled)


def _is_stock(vec):
    return np.all((np.abs(vec) == 1.0) + (np.abs(vec) == 0.0))


def gen_basket_from_returns(returns_flat, seed):
    """Gen_StaticPort logic; sparse entries sampled from `returns_flat` (real returns)."""
    _rng = np.random.RandomState(seed)
    _rvs = lambda size: _rng.choice(returns_flat, size=size, replace=True)   # <-- real values
    _unscale = sparse.random(_N_STOCKS, max(int(_N_STOCKS ** 2), _N_PORTS),
                             density=0.9, data_rvs=_rvs, random_state=_rng).toarray()
    _scale = _unscale / (np.abs(_unscale).sum(0) + 1e-12)
    _price_start = np.ones(_N_STOCKS)                       # same as Gen_StaticPort
    _position = np.diag(1.0 / _price_start.dot(np.abs(_scale)))
    _trans = np.dot(_scale, _position)
    return _trans[:, ~np.apply_along_axis(_is_stock, 0, _trans)]


# The five baskets the evaluation scores against, pinned by seed instead of picked
# out of a candidate pool, so a fresh clone rebuilds exactly these five. The seeds
# are the indices the twenty candidate draw selected; each basket takes its IS
# matrix from the seed itself and its OOS matrix from 100 + seed, which is how the
# pool was laid out. Changing this list changes every reported VaR/ES number,
# since fifty of Compute_PNL's sixty eight columns are these portfolios.
# Seed 0 is replaced by 1 because its OOS draw contains a portfolio whose real VaR
# sits at 0.0084 against a median of 1.9 over the sixty eight P&L columns, and the
# relative error divides by that, so one column would dominate every VaR and ES
# entry. The screen is min|real VaR| over the columns >= 0.15, which seeds 101,
# 105, 107, 115 and 119 all clear; it needs Compute_PNL, so it is applied once and
# recorded here rather than run inside this notebook.
SELECTED_SEEDS = [1, 19, 5, 15, 7, 2, 8, 9, 10, 13]

_base = os.path.join("data", "our_basket")
shutil.rmtree(_base, ignore_errors=True)
for _b, _s in enumerate(SELECTED_SEEDS):
    _d = os.path.join(_base, f"basket{_b}"); os.makedirs(_d, exist_ok=True)
    _W_is = gen_basket_from_returns(_train_flat, seed=_s)         # IS  <- real TRAIN returns
    _W_oos = gen_basket_from_returns(_test_flat, seed=100 + _s)   # OOS <- real TEST  returns
    np.save(os.path.join(_d, "TransMat_IS.npy"), _W_is)
    np.save(os.path.join(_d, "TransMat_OOS.npy"), _W_oos)
    print(f"  basket{_b}: seed {_s}/{100 + _s}  "
          f"IS {_W_is.shape} (train) | OOS {_W_oos.shape} (test)")
print(f"data/our_basket built from real train/test log returns, seeds {SELECTED_SEEDS}")
